# Three firm pairs: an interpretable cointegration study

Read in order: inspect prices, assess integration, estimate a fixed linear relationship, inspect its spread, and simulate a fully specified trading rule. The baseline estimation, half-life window, signals, execution delay and costs are retained from the original notebook so the effect of this review is understandable.

The main analysis uses **one fixed OLS hedge**. Rolling OLS and Kalman experiments, original results, figures and references remain intact in [the original notebook](history/2026-11-09_before_review/Cointegration_Testing_3_Firms.ipynb). They are not ranked in this workflow. Kalman is a specified statistical model, but its extra assumptions and different signal made the earlier comparison harder to interpret.

Every execution saves prices, assumptions and a daily accounting ledger under `runs/`. Set `REPLAY_RUN` in the loading cell to an archived run directory to reuse prices. Earlier outputs alone do not reconstruct the original price inputs; these were not saved by the old notebook. See [REVIEW.md](REVIEW.md) for findings and version retrieval.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import importlib.metadata
import numpy as np
import yfinance as yf
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from IPython.display import display

### Loading the data 

In [2]:
pairs = [("KO", "PEP"), ("V", "MA"), ("SHEL", "BP")]
start, split_date, end = "2016-10-20", "2022-10-20", "2026-10-20"
# Set to an earlier runs/<id> directory to reuse its exact downloaded data.
REPLAY_RUN = None
run_dir = Path("runs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
run_dir.mkdir(parents=True, exist_ok=False)
# Keep Yahoo cache within this writable run directory.
yf.set_tz_cache_location(str(run_dir / "provider_cache"))
pair_data, quality_rows, input_hashes = {}, [], {}
for pair in pairs:
    filename = "-".join(pair) + ".csv"
    if REPLAY_RUN is None:
        raw = yf.download(list(pair), start=start, end=end,
                          auto_adjust=True, progress=False)["Close"]
    else:
        raw = pd.read_csv(Path(REPLAY_RUN) / filename, index_col=0, parse_dates=True)
    raw = raw.loc[:, list(pair)].sort_index()
    if raw.empty or raw.index.has_duplicates:
        raise ValueError(f"Empty data or duplicate dates for {pair}")
    raw.to_csv(run_dir / filename, float_format="%.17g")
    input_hashes[filename] = hashlib.sha256((run_dir / filename).read_bytes()).hexdigest()
    complete = raw.dropna(how="any")
    if not np.isfinite(complete.to_numpy()).all() or (complete <= 0).any().any():
        raise ValueError(f"Nonfinite or nonpositive prices for {pair}")
    pair_data[pair] = {"train": complete.loc[complete.index < split_date],
                       "test": complete.loc[complete.index >= split_date]}
    if min(len(frame) for frame in pair_data[pair].values()) < 30:
        raise ValueError(f"At least 30 train and test observations required for {pair}")
    for ticker in pair:
        quality_rows.append({"Pair": "-".join(pair), "Ticker": ticker,
                             "Downloaded rows": len(raw),
                             "Missing before removal": int(raw[ticker].isna().sum()),
                             "Rows removed for pair": len(raw) - len(complete)})
quality = pd.DataFrame(quality_rows)
display(quality)
print("Run archive:", run_dir.resolve())


1 Failed download:
['KO']: OperationalError('database is locked')


ValueError: At least 30 train and test observations required for ('KO', 'PEP')

In [ ]:
# A table makes the nested data structure visible without drawing machinery.
display(pd.DataFrame([
    {"Pair": "-".join(pair), "Sample": sample, "Rows": len(frame),
     "Start": frame.index.min(), "End": frame.index.max()}
    for pair, data in pair_data.items() for sample, frame in data.items()
]))

### Exploratory data analysis (EDA)

Use the **training data only** to inspect data quality, price movements and stationarity before cointegration testing. Prices are already adjusted for splits and dividends (`auto_adjust=True`). The loader reports missing values before removal and saves the downloaded tables. Missing counts below describe the retained training sample. Dates absent from the provider response cannot be counted without an exchange calendar.

ADF has a unit-root null: a small p-value supports stationarity. Failure to reject in price levels and rejection in first differences is consistent with I(1), not proof. These tests use an intercept and no deterministic time trend. Correlation alone is not cointegration. All subsequent parameter choices use training observations only.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from IPython.display import display

# 1. Summarize the retained training data and check integration order.
eda_rows = []
for pair, data in pair_data.items():
    train = data["train"]
    if train.empty:
        raise ValueError(f"No training observations for {pair}")

    for ticker in pair:
        prices = train[ticker]
        eda_rows.append({
            "Ticker": ticker,
            "Start": train.index.min().date(),
            "End": train.index.max().date(),
            "Observations": len(prices),
            "Missing": int(prices.isna().sum()),
            "Duplicate dates": int(train.index.duplicated().sum()),
            "Nonpositive prices": int((prices <= 0).sum()),
            "Min price": prices.min(),
            "Max price": prices.max(),
            "ADF p (levels)": adfuller(prices, regression="c", autolag="AIC")[1],
            "ADF p (differences)": adfuller(prices.diff().dropna(), regression="c", autolag="AIC")[1],
        })

display(pd.DataFrame(eda_rows).set_index("Ticker").round(4))

# 2. Compare price trends and look for large daily moves or volatility changes.
fig, axes = plt.subplots(len(pair_data), 2, figsize=(13, 3.5 * len(pair_data)), squeeze=False)
for row, (pair, data) in enumerate(pair_data.items()):
    train = data["train"].loc[:, list(pair)]
    normalized = train.div(train.iloc[0]).mul(100)
    returns = train.pct_change(fill_method=None).dropna()

    normalized.plot(ax=axes[row, 0], linewidth=1)
    axes[row, 0].set_title(f"{' / '.join(pair)}: adjusted prices (start = 100)")
    axes[row, 0].set_ylabel("Normalized price")

    returns.mul(100).plot(ax=axes[row, 1], linewidth=0.7, alpha=0.75)
    axes[row, 1].set_title(f"Daily returns | correlation = {returns[pair[0]].corr(returns[pair[1]]):.2f}")
    axes[row, 1].set_ylabel("Daily return (%)")
    axes[row, 1].axhline(0, color="grey", linewidth=0.5)
    for ax in axes[row]:
        ax.set_xlabel("Date")
        ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

### Performing the cointegration test

In [ ]:
ols_models = {}
train_spreads = {}
rows = []

for pair, data in pair_data.items():
    train = data["train"]

    y = train[pair[0]]
    x = train[pair[1]]
    ols = sm.OLS(y, sm.add_constant(x)).fit()

    test_stat, p_value, critical_values = coint(
        y, x, trend="c", method="aeg", autolag="aic"
    )

    ols_models[pair] = ols
    train_spreads[pair] = ols.resid.rename("spread")

    rows.append({
        "pair": f"{pair[0]}-{pair[1]}",
        "intercept": ols.params["const"],
        "beta": ols.params[pair[1]],
        "r_squared": ols.rsquared,
        "EG_p_value": p_value,
        "EG_statistic": test_stat,
        **{f"EG critical {level}": value for level, value in zip(["1%", "5%", "10%"], critical_values)},
    })

summary = pd.DataFrame(rows).set_index("pair")
summary["EG_p_Holm"] = multipletests(summary["EG_p_value"], method="holm")[1]
summary = summary.sort_values("EG_p_value")
print(summary)

### Training spread diagnostics

The model is `price A = intercept + beta * price B + spread`. OLS estimates an association, not causality. Beta means shares of B per share of A; the intercept affects the signal but is not a holding. High price R-squared does not establish cointegration.

Engle-Granger tests the null of no cointegration, assuming I(1) inputs. Both raw and Holm-adjusted p-values are shown for the three baseline tests. The reverse V-MA regression below is an additional exploratory diagnostic outside this correction. V-MA remains the previously investigated case study, not a newly selected unseen experiment.

Reference: [statsmodels cointegration documentation](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.coint.html).

In [ ]:
fig, axes = plt.subplots(len(train_spreads), 1, figsize=(12, 8), squeeze=False)
for ax, (pair, spread) in zip(axes[:, 0], train_spreads.items()):
    name = "-".join(pair)
    p_value = summary.loc[name, "EG_p_value"]
    spread.plot(ax=ax, linewidth=1)
    ax.axhline(0, color="grey", linestyle="--")
    ax.set_title(f"{name}: training spread | EG p = {p_value:.4f}")
    ax.set_ylabel("Adjusted-price spread")
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### V–MA: reverse regression and half-life

Chan, Chapter 2, pp. 46–48 (Example 2.4), estimates `change in spread = constant + lambda * lagged spread + error`, with approximate OU half-life `-log(2)/lambda`. We show this alongside the exact AR(1) half-life `-log(2)/log(1 + lambda)`. The half-life belongs to the spread, not the fixed hedge-ratio beta. Neither half-life nor a cointegration p-value guarantees profitable trades.

The initial rolling window is the rounded **training OU half-life**, at least two observations. It is fixed before testing, with no performance optimization. A reverse regression is a diagnostic, not an independent confirmation; its p-value is also unadjusted for multiple testing.

In [ ]:
import numpy as np

pair = ("V", "MA")
train_vm = pair_data[pair]["train"].loc[:, list(pair)].copy()
test_vm = pair_data[pair]["test"].loc[:, list(pair)].copy()
if train_vm.empty or test_vm.empty:
    raise ValueError("Both training and test samples are required.")
vm = pd.concat([train_vm, test_vm]).sort_index()
assert vm.index.is_unique and train_vm.index.max() < test_vm.index.min()
assert np.isfinite(vm.to_numpy()).all() and (vm > 0).all().all()

reverse_model = sm.OLS(train_vm["MA"], sm.add_constant(train_vm[["V"]])).fit()
reverse_p = coint(train_vm["MA"], train_vm["V"], trend="c", autolag="aic")[1]
direction_check = pd.DataFrame([
    {"Direction": "V on MA (baseline)", "Beta": ols_models[pair].params["MA"],
     "EG p-value": summary.loc["V-MA", "EG_p_value"]},
    {"Direction": "MA on V (diagnostic)", "Beta": reverse_model.params["V"],
     "EG p-value": reverse_p},
]).set_index("Direction")
display(direction_check.round(6))

v_ma_spread = train_spreads[pair]
hl_data = pd.DataFrame({"change": v_ma_spread.diff(),
                        "lag": v_ma_spread.shift(1)}).dropna()
half_life_model = sm.OLS(hl_data["change"], sm.add_constant(hl_data[["lag"]])).fit()
lambda_ = float(half_life_model.params["lag"])
phi = 1 + lambda_
if not 0 < phi < 1:
    raise ValueError("Training spread does not support a positive, decaying AR(1) half-life.")
v_ma_half_life = -np.log(2) / np.log(phi)
chan_half_life = -np.log(2) / lambda_
lookback = max(2, int(np.floor(chan_half_life + 0.5)))
if lookback >= len(train_vm):
    raise ValueError("Insufficient training data for the half-life-based window.")
display(half_life_model.summary())
print(f"lambda = {lambda_:.5f}; phi = {phi:.5f}")
print(f"Chan OU half-life: {chan_half_life:.2f}; exact AR(1): {v_ma_half_life:.2f} observations")
print(f"Fixed initial lookback: {lookback} trading observations")

### Rolling Z-scores and trading signals

Chan, Chapter 3, pp. 70–72 (Example 3.2): enter long spread below Z = −1, short spread above Z = +1, and exit at/crossing zero. A long spread buys V and sells beta shares of MA for each V share. The intercept enters the signal, not the tradable holdings. Opposite-band jumps can reverse a position.

The baseline uses the training OLS intercept and beta throughout. Rolling mean and standard deviation use data available through each signal date, including that close; trailing training observations warm up the first test signal. Test positions start flat. A signal at close t executes at close t+1 and first earns P&L from t+1 to t+2. This extra delay avoids assuming an already-observed closing price can also be the fill price.

In [ ]:
ENTRY_Z = 1.0
COST_BPS = 5.0  # per dollar traded, each leg; includes assumed fees/slippage
BORROW_RATE = 0.02  # assumed annual short-borrow rate, not a measured quote
INITIAL_CAPITAL = 100_000.0
static_alpha = float(ols_models[pair].params["const"])
static_beta = float(ols_models[pair].params["MA"])
static_spread = vm["V"] - static_alpha - static_beta * vm["MA"]
spread_mean = static_spread.rolling(lookback).mean()
spread_std = static_spread.rolling(lookback).std(ddof=1).replace(0, np.nan)
static_z = (static_spread - spread_mean) / spread_std

def positions_from_z(z):
    """State at each signal close; start flat and exit at zero."""
    position = 0
    signals = []
    for value in z:
        if not np.isfinite(value):
            position = 0
        else:
            if (position == 1 and value >= 0) or (position == -1 and value <= 0):
                position = 0
            if position == 0:
                if value < -ENTRY_Z:
                    position = 1
                elif value > ENTRY_Z:
                    position = -1
        signals.append(position)
    return pd.Series(signals, index=z.index, name="signal")

static_signals = positions_from_z(static_z.loc[test_vm.index])
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
static_z.loc[test_vm.index].plot(ax=axes[0], linewidth=1, label="Static OLS Z-score")
for level in [-ENTRY_Z, 0, ENTRY_Z]:
    axes[0].axhline(level, color="grey", linestyle="--", linewidth=0.8)
static_signals.plot(ax=axes[1], drawstyle="steps-post")
axes[0].set_title("V-MA: test Z-score and signals (execution delayed one close)")
axes[1].set_ylabel("Signal: -1 / 0 / +1")
plt.tight_layout()
plt.show()

### Backtest on the reserved test period

This is a daily, self-financing share-accounting simulation: P&L comes from the two held legs, not percentage changes in the spread or changes in beta. At each execution close, holdings are rebalanced to approximately 100% gross exposure relative to current equity; this is a beta hedge, not necessarily dollar-neutral. Cash earns zero and there is no additional leverage. Turnover includes entry, daily resizing/rehedging, exits, reversals and final liquidation. Costs apply to both legs; short borrow accrues over calendar days, including weekends. Sharpe assumes a zero risk-free rate and 252 observations/year.

Adjusted closes approximate total-return leg P&L, including short dividend exposure; actual fills, borrow availability, changing borrow fees, taxes and financing are not modeled. Fractional adjusted-price units are allowed. The loader's end date is exclusive: the actual last available observation is printed below. No stop loss or holding-time optimization is added. This remains a research simulation, and this test set must not subsequently be treated as untouched after examining these results.

In [ ]:
def backtest_pair(prices, z, betas, cost_bps=COST_BPS, borrow_rate=BORROW_RATE):
    if prices.empty or not prices.index.is_unique or not prices.index.is_monotonic_increasing:
        raise ValueError("Prices must have unique, increasing dates and at least one row.")
    if not np.isfinite(prices[["V", "MA"]].to_numpy()).all() or (prices[["V", "MA"]] <= 0).any().any():
        raise ValueError("Prices must be positive and finite.")
    if cost_bps < 0 or borrow_rate < 0:
        raise ValueError("Costs and borrow rate must be nonnegative.")
    signals = positions_from_z(z.reindex(prices.index))
    executed = signals.shift(1).fillna(0).astype(int)
    execution_beta = betas.reindex(prices.index).shift(1)
    shares = np.zeros(2)
    equity = INITIAL_CAPITAL
    rows = []
    closes = prices[["V", "MA"]].to_numpy()
    for i, date in enumerate(prices.index):
        previous_equity = equity
        previous_close = closes[i - 1] if i else closes[i]
        pnl = float(shares @ (closes[i] - previous_close))
        days = (date - prices.index[i - 1]).days if i else 0
        borrow = float(np.maximum(-shares * previous_close, 0).sum()) * borrow_rate * days / 365
        equity += pnl - borrow
        if equity <= 0:
            raise ValueError("Equity exhausted; stop the backtest.")
        side = int(executed.iloc[i])
        beta = execution_beta.iloc[i]
        target = np.zeros(2)
        # Liquidate at the final observed close, with closing costs.
        if side and np.isfinite(beta) and i < len(prices) - 1:
            unit = np.array([1.0, -beta])
            # Reserve a small cost buffer when sizing gross exposure.
            target = side * unit * equity / (1 + 2 * cost_bps / 10_000) / (np.abs(unit) @ closes[i])
        traded = float(np.abs(target - shares) @ closes[i])
        cost = traded * cost_bps / 10_000
        equity -= cost
        if equity <= 0:
            raise ValueError("Transaction costs exhausted equity.")
        actual_side = side if np.any(target) else 0
        rows.append({"equity": equity, "return": equity / previous_equity - 1,
                     "pnl": pnl, "transaction_cost": cost, "borrow_cost": borrow,
                     "turnover": traded / previous_equity, "position": actual_side,
                     "signal_at_close": int(signals.iloc[i]),
                     "executed_signal": side, "execution_beta": beta,
                     "held_V_before_close": shares[0], "held_MA_before_close": shares[1],
                     "held_V_after_close": target[0], "held_MA_after_close": target[1],
                     "traded_dollars": traded, "calendar_days": days,
                     "gross_exposure": np.abs(target) @ closes[i] / equity})
        shares = target
    result = pd.DataFrame(rows, index=prices.index)
    result["drawdown"] = result["equity"] / result["equity"].cummax().clip(lower=INITIAL_CAPITAL) - 1
    return result

def performance(result):
    r = result["return"]
    previous = result["position"].shift(1).fillna(0)
    entries = (result["position"].ne(0) & result["position"].ne(previous)).sum()
    closed = (previous.ne(0) & result["position"].ne(previous)).sum()
    return pd.Series({
        "Total return": result["equity"].iloc[-1] / INITIAL_CAPITAL - 1,
        "CAGR (252 obs/year)": (result["equity"].iloc[-1] / INITIAL_CAPITAL) ** (252 / len(r)) - 1,
        "Annual volatility": r.std(ddof=1) * np.sqrt(252),
        "Sharpe (rf=0)": r.mean() / r.std(ddof=1) * np.sqrt(252) if r.std(ddof=1) > 0 else np.nan,
        "Max drawdown": result["drawdown"].min(),
        "Entries": entries, "Closed trades": closed,
        "Time invested": result["position"].ne(0).mean(),
        "Transaction costs ($)": result["transaction_cost"].sum(),
        "Borrow costs ($)": result["borrow_cost"].sum(),
    })

static_betas = pd.Series(static_beta, index=vm.index)
baseline = backtest_pair(test_vm, static_z, static_betas)
print(f"Actual test observations: {test_vm.index.min().date()} to {test_vm.index.max().date()} ({len(test_vm)} rows)")
display(performance(baseline).to_frame("Static OLS baseline").round(4))
# Cost scenarios are diagnostics; do not pick parameters using these test results.
cost_sensitivity = pd.DataFrame({
    f"{bps} bps/leg": performance(backtest_pair(test_vm, static_z, static_betas, cost_bps=bps))
    for bps in [0, 5, 10]
})
display(cost_sensitivity.round(4))

### Inspect and save the baseline result

The following tables and figures are generated from this execution; no historical performance numbers are copied into the narrative. The ledger shows the signal, delayed execution, holdings, turnover, P&L and each cost separately. The old reserved period has already been inspected. These results cannot serve as a fresh validation set.

The half-life rule is a modeling assumption, not an optimal window established by this study. ADF and cointegration tests do not establish profitability. Cost scenarios retain identical signals and assumptions except for transaction costs; equity-dependent sizing can change dollar P&L. No parameter is selected from these scenarios.

In [ ]:
display(baseline.head(10).round(4))
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
static_spread.loc[test_vm.index].plot(ax=axes[0], label="Fixed OLS spread")
axes[0].axhline(0, color="grey", linestyle="--")
axes[0].set_ylabel("Spread (USD)")
(baseline["equity"] / INITIAL_CAPITAL).plot(ax=axes[1])
axes[1].set_ylabel("Equity / initial")
baseline["drawdown"].mul(100).plot(ax=axes[2])
axes[2].set_ylabel("Drawdown (%)")
for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# Reconcile accounting explicitly before writing the result archive.
net_change = baseline["pnl"].sum() - baseline["transaction_cost"].sum() - baseline["borrow_cost"].sum()
np.testing.assert_allclose(baseline["equity"].iloc[-1], INITIAL_CAPITAL + net_change)
assumptions = {
    "pairs": pairs, "start": start, "split_date": split_date, "end_exclusive": end,
    "entry_z": ENTRY_Z, "cost_bps_per_leg": COST_BPS, "annual_borrow_rate": BORROW_RATE,
    "initial_capital": INITIAL_CAPITAL, "alpha": static_alpha, "beta": static_beta,
    "lookback": lookback, "OU_half_life": float(chan_half_life),
    "AR1_half_life": float(v_ma_half_life), "replay_source": str(REPLAY_RUN),
    "input_sha256": input_hashes,
    "notebook_sha256": hashlib.sha256(Path("Cointegration_Testing_3_Firms.ipynb").read_bytes()).hexdigest(),
    "versions": {name: importlib.metadata.version(name) for name in
                 ["numpy", "pandas", "statsmodels", "yfinance", "matplotlib"]},
    "timing": "signal close t; execute close t+1; first P&L close t+2",
}
(run_dir / "assumptions.json").write_text(json.dumps(assumptions, indent=2), encoding="utf-8")
quality.to_csv(run_dir / "data_quality.csv", index=False)
summary.to_csv(run_dir / "cointegration.csv")
pd.DataFrame(eda_rows).to_csv(run_dir / "training_eda.csv", index=False)
direction_check.to_csv(run_dir / "direction_check.csv")
baseline.to_csv(run_dir / "baseline_daily_ledger.csv")
performance(baseline).to_csv(run_dir / "baseline_performance.csv")
cost_sensitivity.to_csv(run_dir / "cost_sensitivity.csv")
pd.DataFrame({"spread": static_spread, "mean": spread_mean,
              "std": spread_std, "z": static_z}).to_csv(run_dir / "signal_inputs.csv")
print("Results saved to", run_dir.resolve())
print("Baseline net return:", f"{performance(baseline)['Total return']:.2%}")
print("Training evidence and simulated profitability must be assessed separately.")


### Interpretation and next evidence

Review missing data, ADF results, adjusted cointegration p-values, OLS coefficients and the spread before interpreting simulated returns. Inspect the daily ledger and sensitivity to costs. A fixed beta improves traceability but can become economically stale. The simulation still uses adjusted-price units, daily resizing, zero cash interest, assumed borrow and closing-price execution; tradable fills and availability require separate evidence.

The archived notebook retains the original empirical conclusions and the rolling/Kalman comparison. Its output is a historical snapshot, not a current recommendation. Its named book references were carried over from that analysis and were not independently checked against a supplied PDF in this review. Any next model comparison needs a fresh period or a prespecified evaluation design, without optimizing on the already inspected period.